# Question 3: RAG vs Fine-Tuning: Decision and Demo

This question tests conceptual depth and your ability to think like a system

designer. You'll write a decision framework and build a small demonstration to

back it up.

**Part A: The Decision Tree**

This function evaluates the constraints of a project (budget, data velocity, latency, and knowledge type) and outputs the most viable architectural approach.

In [ ]:
def recommend_approach(scenario: dict) -> str:
    """
    Evaluates system requirements and recommends an LLM architecture approach.
    """
    data_freq = scenario.get('data_changes_frequently')
    format_req = scenario.get('need_specific_output_format')
    budget = scenario.get('budget')
    latency = scenario.get('latency_sensitive')
    know_type = scenario.get('knowledge_type')

    # 1. RAG + Fine-Tuning
    if know_type == 'both' and budget in ['medium', 'high']:
        return "RAG + Fine-Tuning: Combines real-time factual grounding (RAG) with specialized tone/format training (Fine-Tuning)."

    # 2. Prompt Engineering Only
    if budget == 'low' and not data_freq and not format_req:
        return "Prompt Engineering only: Best for tight budgets when data is static and formats are flexible."

    # 3. Fine-Tuning
    if know_type == 'behavioral' or (format_req and not data_freq):
        if budget == 'low':
             return "Prompt Engineering only: Budget restricts fine-tuning, so behavior must be guided via few-shot prompting."
        if latency:
             return "Fine-Tuning: Bakes the behavior into the model weights, reducing prompt size and improving inference speed."
        return "Fine-Tuning: Ideal for teaching the model a specific behavior, tone, or strict output format that doesn't change."

    # 4. RAG
    if know_type == 'factual' or data_freq:
        if latency and budget == 'high':
             return "RAG + Fine-Tuning: Fine-tune a smaller, faster model to process RAG contexts to hit latency targets."
        return "RAG: The standard approach for querying frequently updated or proprietary factual data without retraining."

    return "Prompt Engineering only: The safest baseline when requirements are vague or resources are strictly limited."


In [ ]:
scenarios = [
    {
        "name": "Customer Support Bot (Dynamic Docs)",
        "data_changes_frequently": True, "need_specific_output_format": False,
        "budget": "medium", "latency_sensitive": False, "knowledge_type": "factual"
    },
    {
        "name": "Medical JSON Extractor",
        "data_changes_frequently": False, "need_specific_output_format": True,
        "budget": "high", "latency_sensitive": True, "knowledge_type": "behavioral"
    },
    {
        "name": "Enterprise Legal Copilot",
        "data_changes_frequently": True, "need_specific_output_format": True,
        "budget": "high", "latency_sensitive": False, "knowledge_type": "both"
    },
    {
        "name": "Student Hobby Project",
        "data_changes_frequently": False, "need_specific_output_format": False,
        "budget": "low", "latency_sensitive": False, "knowledge_type": "factual"
    },
    {
        "name": "Branded Marketing Generator",
        "data_changes_frequently": False, "need_specific_output_format": False,
        "budget": "medium", "latency_sensitive": False, "knowledge_type": "behavioral"
    }
]

print("--- Architecture Decision Results ---")
for s in scenarios:
    print(f"\nScenario: {s['name']}")
    print(f"Recommendation: {recommend_approach(s)}")

--- Architecture Decision Results ---

Scenario: Customer Support Bot (Dynamic Docs)
Recommendation: RAG: The standard approach for querying frequently updated or proprietary factual data without retraining.

Scenario: Medical JSON Extractor
Recommendation: Fine-Tuning: Bakes the behavior into the model weights, reducing prompt size and improving inference speed.

Scenario: Enterprise Legal Copilot
Recommendation: RAG + Fine-Tuning: Combines real-time factual grounding (RAG) with specialized tone/format training (Fine-Tuning).

Scenario: Student Hobby Project
Recommendation: Prompt Engineering only: Best for tight budgets when data is static and formats are flexible.

Scenario: Branded Marketing Generator
Recommendation: Fine-Tuning: Ideal for teaching the model a specific behavior, tone, or strict output format that doesn't change.
